# ✈️ Flight Delays and Cancellations | Feature Engineering

**Input:** `data/raw/flights.csv` · `data/raw/airlines.csv` · `data/raw/airports.csv`  
**Output:** `data/processed/flights_features.parquet`

## Objective

Transformar o dataset bruto de voos em um conjunto de features prontas para os
modelos de **classificação** (atraso ≥ 15 min?) e **clustering** (perfil de causa
de atraso), cobrindo:

- Tratamento de nulos e remoção de escopo (cancelados / desviados).
- Criação de features temporais, de rota e históricas.
- Prevenção de **data leakage** nas features históricas (split antes dos agregados).
- Encoding de variáveis categóricas (fit exclusivo no treino).
- Exportação em Parquet particionado.

## Scope

- Apenas voos **concluídos** (`CANCELLED = 0`, `DIVERTED = 0`) com `ARRIVAL_DELAY` preenchido.
- Dataset: voos domésticos dos EUA — ano 2015 (Bureau of Transportation Statistics).
- Split temporal: **treino** = meses 1-10 · **teste** = meses 11-12 (consistente com
  o notebook de classificação).

## Setup

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, FloatType, DoubleType
)
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline

import warnings

warnings.filterwarnings('ignore')
os.makedirs('../data/processed', exist_ok=True)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

In [ ]:
os.environ['JAVA_HOME'] = "/usr/lib/jvm/java-17-openjdk-amd64"
spark = (
    SparkSession.builder
        .appName("FeatureEngineering")
        .master("local[*]")
        .config("spark.driver.memory", "4g")
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.sql.repl.eagerEval.enabled", True)
        .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("="*80)
print(f"Spark Session iniciada ...\n\tSpark Version: {spark.version}\n\tPython Version: {sys.version}")
print("="*80)

In [ ]:
PATH_FLIGHTS  = '../data/raw/flights.csv'
PATH_AIRLINES = '../data/raw/airlines.csv'
PATH_AIRPORTS = '../data/raw/airports.csv'
PATH_OUTPUT   = '../data/processed/flights_features.parquet'

os.makedirs('../data/processed', exist_ok=True)

## Data Loading

In [ ]:
# ── Schema explícito ─────────────────────────────────────────────────────────
# Evita inferência incorreta de tipos e melhora a performance na leitura.
SCHEMA_FLIGHTS = StructType([
    StructField('YEAR',                 IntegerType(), True),
    StructField('MONTH',                IntegerType(), True),
    StructField('DAY',                  IntegerType(), True),
    StructField('DAY_OF_WEEK',          IntegerType(), True),
    StructField('AIRLINE',              StringType(),  True),
    StructField('FLIGHT_NUMBER',        IntegerType(), True),
    StructField('TAIL_NUMBER',          StringType(),  True),
    StructField('ORIGIN_AIRPORT',       StringType(),  True),
    StructField('DESTINATION_AIRPORT',  StringType(),  True),
    StructField('SCHEDULED_DEPARTURE',  IntegerType(), True),
    StructField('DEPARTURE_TIME',       FloatType(),   True),
    StructField('DEPARTURE_DELAY',      FloatType(),   True),
    StructField('TAXI_OUT',             FloatType(),   True),
    StructField('WHEELS_OFF',           FloatType(),   True),
    StructField('SCHEDULED_TIME',       FloatType(),   True),
    StructField('ELAPSED_TIME',         FloatType(),   True),
    StructField('AIR_TIME',             FloatType(),   True),
    StructField('DISTANCE',             FloatType(),   True),
    StructField('WHEELS_ON',            FloatType(),   True),
    StructField('TAXI_IN',              FloatType(),   True),
    StructField('SCHEDULED_ARRIVAL',    IntegerType(), True),
    StructField('ARRIVAL_TIME',         FloatType(),   True),
    StructField('ARRIVAL_DELAY',        FloatType(),   True),
    StructField('DIVERTED',             IntegerType(), True),
    StructField('CANCELLED',            IntegerType(), True),
    StructField('CANCELLATION_REASON',  StringType(),  True),
    StructField('AIR_SYSTEM_DELAY',     FloatType(),   True),
    StructField('SECURITY_DELAY',       FloatType(),   True),
    StructField('AIRLINE_DELAY',        FloatType(),   True),
    StructField('LATE_AIRCRAFT_DELAY',  FloatType(),   True),
    StructField('WEATHER_DELAY',        FloatType(),   True),
])

df = (
    spark.read
    .option('header', 'true')
    .option('nullValue', 'NA')
    .schema(SCHEMA_FLIGHTS)
    .csv(PATH_FLIGHTS)
)

df_airlines = spark.read.option('header', 'true').csv(PATH_AIRLINES)
df_airports = spark.read.option('header', 'true').csv(PATH_AIRPORTS)

print(f'Registros carregados: {df.count():,}')
print(f'Colunas: {len(df.columns)}')

## Scope Definition

> O modelo supervisionado deve prever o atraso na **chegada** dos voos que
> **foram concluídos**. Portanto, voos desviados e cancelados são removidos.
>
> Registros sem `ARRIVAL_DELAY` informado também são descartados.

In [ ]:
n_before = df.count()

df = df.filter(
    (F.col('CANCELLED') == 0) &
    (F.col('DIVERTED')  == 0) &
    (F.col('ARRIVAL_DELAY').isNotNull())
)

n_after = df.count()
print(f'Removidos (cancelados/desviados/sem ARRIVAL_DELAY): {n_before - n_after:,}')
print(f'Registros no escopo do modelo: {n_after:,}')

df = df.drop('CANCELLED', 'DIVERTED', 'CANCELLATION_REASON')
print(f'Colunas restantes: {len(df.columns)}')

## Null Values

### Delay Cause Columns

**Colunas:** `AIR_SYSTEM_DELAY`, `SECURITY_DELAY`, `AIRLINE_DELAY`,
`LATE_AIRCRAFT_DELAY`, `WEATHER_DELAY`

**Hipótese (H₀):**  
O DOT **não** preenche as colunas de causa quando `ARRIVAL_DELAY < 15`.
Portanto, nulos nessas colunas representam ausência de atraso registrado
(equivalente a 0 minutos), e não dados faltantes de verdade.

In [ ]:
DELAY_CAUSE_COLS = [
    'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY',
    'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY'
]

In [ ]:
# ── Verificação da hipótese ──────────────────────────────────────────────────
df_on_time = df.filter(F.col('ARRIVAL_DELAY') <  15)
df_delayed = df.filter(F.col('ARRIVAL_DELAY') >= 15)

n_on_time = df_on_time.count()
n_delayed = df_delayed.count()

print(f'Voos pontuais  (ARRIVAL_DELAY <  15): {n_on_time:>9,}')
print(f'Voos atrasados (ARRIVAL_DELAY >= 15): {n_delayed:>9,}')
print()

for col in DELAY_CAUSE_COLS:
    nulls_on_time = df_on_time.filter(F.col(col).isNull()).count()
    nulls_delayed = df_delayed.filter(F.col(col).isNull()).count()
    print(f'  {col:<22} | pontuais: {nulls_on_time/n_on_time*100:>6.2f}% nulos | '
          f'atrasados: {nulls_delayed/n_delayed*100:>6.2f}% nulos')

In [ ]:
# ── Existem voos pontuais com alguma causa de atraso preenchida? ─────────────
anomaly_condition = ' OR '.join([f'{c} IS NOT NULL' for c in DELAY_CAUSE_COLS])
n_anomalies = df_on_time.filter(anomaly_condition).count()

print(f'Voos pontuais (ARRIVAL_DELAY < 15 min) com causa preenchida: {n_anomalies:,}')
if n_anomalies == 0:
    print('Hipótese confirmada.')
else:
    print(f'Hipótese rejeitada: {n_anomalies:,} registros anormais detectados.')

In [ ]:
# ── Variável alvo do modelo supervisionado ───────────────────────────────────
# LABEL = 1 → atraso na chegada ≥ 15 min  |  LABEL = 0 → pontual
df = df.withColumn(
    'LABEL',
    F.when(F.col('ARRIVAL_DELAY') >= 15, 1).otherwise(0).cast(IntegerType())
)

In [ ]:
# ── Preencher nulos das colunas de causa com 0 ───────────────────────────────
# ARRIVAL_DELAY <  15  →  DOT não registra causa  →  nulo = 0 min
# ARRIVAL_DELAY >= 15  →  DOT registra causa      →  valor real
df = df.fillna(0, subset=DELAY_CAUSE_COLS)

In [ ]:
remaining_nulls = {c: df.filter(F.col(c).isNull()).count() for c in DELAY_CAUSE_COLS}
print('Nulos residuais após fillna(0):', remaining_nulls)

In [ ]:
n_total = df.count()
label_dist = (
    df.groupBy('LABEL').count()
    .withColumn('PCT', F.round(F.col('count') / n_total * 100, 2))
    .orderBy('LABEL')
    .toPandas()
)
label_dist['DESC'] = label_dist['LABEL'].map(
    {0: 'Não atrasado (< 15 min)', 1: 'Atrasado (>= 15 min)'}
)
print('Distribuição final da variável-alvo:')
print(label_dist[['DESC', 'count', 'PCT']].to_string(index=False))
print('\nRatio 0:1 =', round(label_dist.loc[0, 'count'] / label_dist.loc[1, 'count'], 2))

# Colunas de causa são leakage no feature set preditivo
LEAKAGE_COLS = DELAY_CAUSE_COLS.copy()
print(f'\nColunas marcadas como LEAKAGE: {LEAKAGE_COLS}')

### Operational Columns

Como visto na EDA, apenas `ELAPSED_TIME` e `SCHEDULED_TIME` apresentam
percentual de nulos significativo. Registros com nulo nessas colunas são
removidos — representam voos com dados operacionais incompletos.

In [ ]:
n_before = df.count()

df = df.filter(
    F.col('ELAPSED_TIME').isNotNull() &
    F.col('SCHEDULED_TIME').isNotNull()
)

n_after   = df.count()
n_removed = n_before - n_after

print(f'Registros antes : {n_before:,}')
print(f'Registros depois: {n_after:,}')
print(f'Removidos       : {n_removed:,} ({n_removed/n_before*100:.2f}%)')

# ── Verificar nulos residuais nas demais colunas operacionais ─────────────────
OPERATIONAL_COLS = [
    'DEPARTURE_DELAY', 'DEPARTURE_TIME', 'TAXI_OUT', 'TAXI_IN',
    'ELAPSED_TIME', 'AIR_TIME', 'WHEELS_OFF', 'WHEELS_ON',
    'ARRIVAL_TIME', 'SCHEDULED_TIME'
]

print('\nNulos residuais nas colunas operacionais:')
for c in OPERATIONAL_COLS:
    n = df.filter(F.col(c).isNull()).count()
    status = 'OK' if n == 0 else f'ATENCAO  {n:,}'
    print(f'  {c:<22}: {status}')

### Dropping Irrelevant Columns

Colunas sem sinal preditivo generalizável são descartadas antes do
feature engineering para reduzir o volume de dados em memória.

In [ ]:
# ── Colunas a descartar e motivo de cada uma ─────────────────────────────────
DROP_COLS = {
    'YEAR'         : 'Constante (todos os registros são de 2015) — variância zero',
    'TAIL_NUMBER'  : 'Alta cardinalidade sem padrão generalizável por aeronave',
    'FLIGHT_NUMBER': 'Identificador operacional — não carrega sinal preditivo estável',
}

for col, reason in DROP_COLS.items():
    print(f'  DROP | {col:<15} → {reason}')

df = df.drop(*DROP_COLS.keys())
print(f'\nColunas restantes: {len(df.columns)}')
print(df.columns)

In [ ]:
n = df.count()
label_dist = (
    df.groupBy('LABEL').count()
    .withColumn('PCT', F.round(F.col('count') / n * 100, 2))
    .orderBy('LABEL')
    .toPandas()
)
label_dist['DESC'] = label_dist['LABEL'].map(
    {0: 'Pontual (< 15 min)', 1: 'Atrasado (>= 15 min)'}
)
print(f'Total de registros: {n:,}')
print(label_dist[['DESC', 'count', 'PCT']].to_string(index=False))

## Feature Engineering

### Temporal Features

`SCHEDULED_DEPARTURE` e `SCHEDULED_ARRIVAL` estão no formato HHMM
(ex: 830 = 08h30). Dividir por 100 e truncar extrai a hora inteira.

In [ ]:
# ── Hora de partida e chegada programadas ────────────────────────────────────
# Exemplos: 830 → 08h30  |  1245 → 12h45  |  2359 → 23h59
df = df.withColumn('HORA_PARTIDA',
        (F.col('SCHEDULED_DEPARTURE') / 100).cast(IntegerType()))

df = df.withColumn('HORA_CHEGADA_PROG',
        (F.col('SCHEDULED_ARRIVAL') / 100).cast(IntegerType()))

# ── Turno do dia ──────────────────────────────────────────────────────────────
# Captura o padrão bola de neve de atrasos de forma não-linear
df = df.withColumn('TURNO',
    F.when(F.col('HORA_PARTIDA').between(5,  11), F.lit('manha'))
    .when(F.col('HORA_PARTIDA').between(12, 17), F.lit('tarde'))
    .when(F.col('HORA_PARTIDA').between(18, 21), F.lit('noite'))
    .otherwise(F.lit('madrugada'))
)

# ── Flags binários ────────────────────────────────────────────────────────────
df = df.withColumn('IS_WEEKEND',
    F.when(F.col('DAY_OF_WEEK').isin([6, 7]), 1).otherwise(0).cast(IntegerType()))

df = df.withColumn('IS_PEAK_MONTH',
    F.when(F.col('MONTH').isin([6, 7, 12]), 1).otherwise(0).cast(IntegerType()))

# ── Verificação ───────────────────────────────────────────────────────────────
print('Features temporais criadas:')
df.select(
    'SCHEDULED_DEPARTURE', 'HORA_PARTIDA',
    'SCHEDULED_ARRIVAL', 'HORA_CHEGADA_PROG',
    'TURNO', 'IS_WEEKEND', 'IS_PEAK_MONTH'
).show(5, truncate=False)

### Route Features

Origem-destino como identificador único de rota, frequência da rota
(característica estrutural — sem risco de leakage) e log da distância
(reduz o peso de rotas muito longas na distribuição assimétrica).

In [ ]:
# ── Identificador de rota ────────────────────────────────────────────────────
df = df.withColumn('ROTA',
    F.concat(F.col('ORIGIN_AIRPORT'), F.lit('_'), F.col('DESTINATION_AIRPORT')))

# ── Frequência da rota ────────────────────────────────────────────────────────
# Característica estrutural da rota — calculada sobre o dataset completo
# (não depende de ARRIVAL_DELAY, logo sem risco de leakage).
route_freq = (
    df.groupBy('ROTA')
    .count()
    .withColumnRenamed('count', 'FREQ_ROTA')
)
df = df.join(route_freq, on='ROTA', how='left')

# ── Log da distância ──────────────────────────────────────────────────────────
df = df.withColumn('LOG_DISTANCE', F.log1p(F.col('DISTANCE')))

print('Features de rota criadas:')
df.select('ROTA', 'FREQ_ROTA', 'DISTANCE', 'LOG_DISTANCE').show(5)

### Historical Features

Atraso médio histórico por grupo (companhia, aeroporto de origem,
aeroporto de destino, rota).

> **⚠️ Atenção — Data Leakage**  
> Essas features são derivadas de `ARRIVAL_DELAY`, a própria variável alvo.  
> Se calculadas sobre o dataset completo, os dados de **teste (meses 11-12)**
> contaminam as médias usadas no treino — o modelo aprenderia padrões que
> não existiriam no momento real da predição.  
>  
> **Correção:**  
> 1. Separar `df_train` (meses 1-10) e `df_test` (meses 11-12) **antes** do cálculo.  
> 2. Calcular as médias históricas **somente sobre `df_train`**.  
> 3. Aplicar as médias no teste via join (grupos ausentes → imputação pela média
>    global do treino).  
> 4. Reunificar em `df` para que o restante do notebook continue sem alterações.

In [ ]:
# ── Separação temporal ANTES dos agregados (evita data leakage) ─────────────
# O split usa os mesmos limites que o notebook de classificação (meses 1-10 /
# 11-12), garantindo consistência entre as duas etapas do pipeline.
df_train = df.filter(F.col('MONTH') <= 10)
df_test  = df.filter(F.col('MONTH') >  10)

print(f'Treino (meses  1-10): {df_train.count():>10,} registros')
print(f'Teste  (meses 11-12): {df_test.count():>10,} registros')


# ── Função auxiliar ───────────────────────────────────────────────────────────
# df_ref   : fonte dos dados para o cálculo das médias (sempre o treino)
# df_target: dataset que receberá a feature via join
def add_hist_delay(df_ref, df_target, group_cols, new_col):
    if isinstance(group_cols, str):
        group_cols = [group_cols]
    agg = (
        df_ref.groupBy(group_cols)
        .agg(F.round(F.mean('ARRIVAL_DELAY'), 3).alias(new_col))
    )
    return df_target.join(agg, on=group_cols, how='left')


# ── Calcular médias no treino → aplicar em treino e teste ────────────────────
HIST_GROUPS = [
    ('AIRLINE',                  'HIST_DELAY_AIRLINE'),
    ('ORIGIN_AIRPORT',           'HIST_DELAY_ORIGIN'),
    ('DESTINATION_AIRPORT',      'HIST_DELAY_DEST'),
    ('ROTA',                     'HIST_DELAY_ROTA'),
    (['AIRLINE', 'DAY_OF_WEEK'], 'HIST_DELAY_AIRLINE_DOW'),
    (['AIRLINE', 'TURNO'],       'HIST_DELAY_AIRLINE_TURNO'),
]

for group_cols, col_name in HIST_GROUPS:
    df_train = add_hist_delay(df_train, df_train, group_cols, col_name)
    df_test  = add_hist_delay(df_train, df_test,  group_cols, col_name)


# ── Imputação de nulos no teste (grupos ausentes no treino) ──────────────────
# Rotas ou combinações que não existem no treino recebem null após o join.
# Fallback: média global do treino — conservador e livre de leakage.
HIST_COLS = [col for _, col in HIST_GROUPS]
global_means = (
    df_train
    .agg(*[F.mean(c).alias(c) for c in HIST_COLS])
    .first()
    .asDict()
)

for c in HIST_COLS:
    df_test = df_test.withColumn(
        c, F.coalesce(F.col(c), F.lit(global_means[c]))
    )


# ── Reunificar ────────────────────────────────────────────────────────────────
# Após o cálculo correto das features, df volta a ser único para que
# as etapas seguintes (encoding, seleção, exportação) funcionem sem alteração.
df = df_train.union(df_test)

print('\nFeatures históricas criadas (sem leakage):')
df.select(*HIST_COLS).show(5)

## Feature Validation

Verifica que todas as features criadas estão livres de nulos antes
de prosseguir com o encoding e a seleção final.

In [ ]:
NEW_FEATURES = [
    'HORA_PARTIDA', 'HORA_CHEGADA_PROG', 'TURNO', 'IS_WEEKEND', 'IS_PEAK_MONTH',
    'ROTA', 'FREQ_ROTA', 'LOG_DISTANCE',
    'HIST_DELAY_AIRLINE', 'HIST_DELAY_ORIGIN', 'HIST_DELAY_DEST',
    'HIST_DELAY_ROTA', 'HIST_DELAY_AIRLINE_DOW', 'HIST_DELAY_AIRLINE_TURNO'
]

print(f'Features criadas: {len(NEW_FEATURES)}')
print(f'Colunas totais:   {len(df.columns)}\n')

print('Nulos nas novas features:')
for c in NEW_FEATURES:
    n = df.filter(F.col(c).isNull()).count()
    status = 'OK  0' if n == 0 else f'ATENCAO  {n:,}'
    print(f'  {c:<28}: {status}')

## Encoding

Variáveis categóricas são convertidas para índices numéricos com
`StringIndexer`. O mapeamento string→índice é **aprendido exclusivamente
no treino** (`MONTH <= 10`) para evitar que categorias do teste influenciem
a codificação. `handleInvalid='keep'` garante que categorias inéditas no
teste recebam um índice extra em vez de causar erro.

In [ ]:
CATEGORICAL_COLS = ['AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'TURNO']

indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=f'{c}_IDX',
        handleInvalid='keep'  # categoria nova em producao → indice extra, nao quebra
    )
    for c in CATEGORICAL_COLS
]

# Fit apenas no treino
encoder = Pipeline(stages=indexers).fit(df.filter(F.col('MONTH') <= 10))
df = encoder.transform(df)

print('Encoding aplicado:')
df.select([f'{c}_IDX' for c in CATEGORICAL_COLS]).show(5)

## Final Selection & Export

In [ ]:
# ── Features do modelo de classificação ──────────────────────────────────────
CLASSIFIER_FEATURES = [
    # Temporais
    'MONTH', 'DAY_OF_WEEK', 'HORA_PARTIDA', 'HORA_CHEGADA_PROG',
    'IS_WEEKEND', 'IS_PEAK_MONTH',
    # Rota e distância
    'LOG_DISTANCE', 'FREQ_ROTA',
    # Categóricas indexadas
    'AIRLINE_IDX', 'ORIGIN_AIRPORT_IDX', 'DESTINATION_AIRPORT_IDX', 'TURNO_IDX',
    # Históricas
    'HIST_DELAY_AIRLINE', 'HIST_DELAY_ORIGIN', 'HIST_DELAY_DEST',
    'HIST_DELAY_ROTA', 'HIST_DELAY_AIRLINE_DOW', 'HIST_DELAY_AIRLINE_TURNO',
]

# ── Features para o clustering (mantidas para o notebook de clustering) ───────
CLUSTERING_FEATURES = [
    'AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'ROTA',
    'ARRIVAL_DELAY', 'DEPARTURE_DELAY',
    'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY',
    'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY',
]

# ── Seleção final ─────────────────────────────────────────────────────────────
FINAL_COLS = CLASSIFIER_FEATURES + ['LABEL'] + CLUSTERING_FEATURES

df_output = df.select(FINAL_COLS)

print(f'Features do modelo : {len(CLASSIFIER_FEATURES)}')
print(f'Colunas no Parquet : {len(FINAL_COLS)}')

print(f'\nColunas descartadas nesta etapa:')
dropped_cols = [c for c in df.columns if c not in FINAL_COLS]
for c in dropped_cols:
    print(f'  {c}')

In [ ]:
# ── Confirmar zero nulos no feature set do modelo ────────────────────────────
print('Verificando nulos nas features do modelo...')
null_counts = {
    c: df_output.filter(F.col(c).isNull()).count()
    for c in CLASSIFIER_FEATURES
}
issues = {k: v for k, v in null_counts.items() if v > 0}

if not issues:
    print('Zero nulos nas features do modelo\n')
else:
    print(f'Nulos encontrados: {issues}')

# ── Salvar em Parquet ──────────────────────────────────────────────────────────
(
    df_output
    .repartition(4)
    .write
    .mode('overwrite')
    .parquet('../data/processed/flights_features.parquet')
)

# ── Validação pós-escrita ─────────────────────────────────────────────────────
df_validation = spark.read.parquet('../data/processed/flights_features.parquet')
print(f'Registros salvos : {df_validation.count():,}')
print(f'Colunas          : {len(df_validation.columns)}')
print(f'\nSchema final:')
df_validation.printSchema()